In [ ]:
# Binary RNN

In [1]:
import re
import torch
import emoji
import torch.nn as nn
import pandas as pd
import json
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import classification_report

# Load your datasets
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

# Clean the text
def clean_text(text):
    text = re.sub(r"#USER#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower()
    text = convert_emojis(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', 'USER_MENTION', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower().strip()

train_df["clean_text"] = train_df["text"].apply(clean_text)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)

# Map labels
label_map = {"Hope": 1, "Not Hope": 0}
train_df["label"] = train_df["binary"].map(label_map)
dev_df["label"] = dev_df["binary"].map(label_map)

# Vocabulary
word_counts = Counter()
for sentence in train_df["clean_text"]:
    word_counts.update(sentence.split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def text_to_tensor(text, vocab):
    return torch.tensor([vocab.get(word, vocab["<UNK>"]) for word in text.split()], dtype=torch.long)

# Dataset class
class HopeDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = [text_to_tensor(text, vocab) for text in texts]
        self.labels = torch.tensor(labels.values, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return padded_texts, torch.tensor(labels, dtype=torch.float)

train_dataset = HopeDataset(train_df["clean_text"], train_df["label"], vocab)
dev_dataset = HopeDataset(dev_df["clean_text"], dev_df["label"], vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# RNN Model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        out = self.fc(hidden.squeeze(0))
        return self.sigmoid(out)

# Training + Evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNClassifier(len(vocab)).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

def evaluate_model(model, dev_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in dev_loader:
            texts = texts.to(device)
            outputs = model(texts).cpu().squeeze()
            preds = (outputs > 0.5).long()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    print(classification_report(all_labels, all_preds, target_names=["Not Hope", "Hope"]))

# Run training and evaluation
train_model(model, train_loader, criterion, optimizer)
evaluate_model(model, dev_loader)

# Save model and vocab
model_path = "saved_rnn_model.pt"
vocab_path = "saved_vocab.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w") as f:
    json.dump(vocab, f)

print(f"✅ Model saved to '{model_path}'")
print(f"✅ Vocabulary saved to '{vocab_path}'")


Epoch 1/10, Loss: 113.4008
Epoch 2/10, Loss: 112.3393
Epoch 3/10, Loss: 109.1966
Epoch 4/10, Loss: 111.7158
Epoch 5/10, Loss: 94.3863
Epoch 6/10, Loss: 64.9764
Epoch 7/10, Loss: 50.7069
Epoch 8/10, Loss: 37.7265
Epoch 9/10, Loss: 26.4632
Epoch 10/10, Loss: 18.4414
              precision    recall  f1-score   support

    Not Hope       0.79      0.81      0.80      1003
        Hope       0.78      0.76      0.77       899

    accuracy                           0.79      1902
   macro avg       0.79      0.79      0.79      1902
weighted avg       0.79      0.79      0.79      1902

✅ Model saved to 'saved_rnn_model.pt'
✅ Vocabulary saved to 'saved_vocab.json'


In [2]:
# RoBERTa Binary

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Load data
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Label mapping
label_map = {"Hope": 1, "Not Hope": 0}
train_df["label"] = train_df["binary"].map(label_map)
dev_df["label"] = dev_df["binary"].map(label_map)

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Dataset class
class HopeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Prepare datasets
train_dataset = HopeDataset(train_df["text"], train_df["label"], tokenizer)
dev_dataset = HopeDataset(dev_df["text"], dev_df["label"], tokenizer)

# Load model
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

# Compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, output_dict=True)
    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./results_roberta",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",  # Save checkpoint each epoch
    logging_dir="./logs_roberta",
    logging_steps=10,
    save_total_limit=1,  # Keep only the latest checkpoint
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save final model and tokenizer
save_path = "saved_roberta_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model and tokenizer saved to '{save_path}'")


C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ryan\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly 

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [4]:
# GPT2 Binary

In [6]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2Model, GPT2ForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report

# Load datasets
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Map labels
label_map = {"Hope": 1, "Not Hope": 0}
train_df["label"] = train_df["binary"].map(label_map)
dev_df["label"] = dev_df["binary"].map(label_map)

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 doesn't have an official pad token

# Dataset class
class GPT2HopeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = GPT2HopeDataset(train_df["text"], train_df["label"], tokenizer)
dev_dataset = GPT2HopeDataset(dev_df["text"], dev_df["label"], tokenizer)

# Load GPT-2 model with classification head
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id  # Prevents warning

# Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, output_dict=True)
    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./results_gpt2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_gpt2",
    logging_steps=10,
    save_total_limit=1,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train and evaluate
trainer.train()
trainer.evaluate()

# Save final model and tokenizer
save_path = "saved_gpt2_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ GPT-2 model saved to '{save_path}'")


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Ryan\AppData\Local\Temp\ipykernel_13996\1669374065.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [7]:
# BiLSTM + Attention model Binary

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import re
import emoji
import json
from collections import Counter
from sklearn.metrics import classification_report

# Load data
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Text preprocessing
def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def clean_text(text):
    text = re.sub(r"#USER#", "", text)
    text = convert_emojis(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', 'USER_MENTION', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

train_df["clean_text"] = train_df["text"].apply(clean_text)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)

# Label mapping
label_map = {"Hope": 1, "Not Hope": 0}
train_df["label"] = train_df["binary"].map(label_map)
dev_df["label"] = dev_df["binary"].map(label_map)

# Vocabulary
word_counts = Counter()
for sentence in train_df["clean_text"]:
    word_counts.update(sentence.split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def text_to_tensor(text, vocab):
    return torch.tensor([vocab.get(word, vocab["<UNK>"]) for word in text.split()], dtype=torch.long)

# Dataset and dataloader
class HopeDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = [text_to_tensor(text, vocab) for text in texts]
        self.labels = torch.tensor(labels.values, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return padded_texts, torch.tensor(labels, dtype=torch.float)

train_dataset = HopeDataset(train_df["clean_text"], train_df["label"], vocab)
dev_dataset = HopeDataset(dev_df["clean_text"], dev_df["label"], vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# BiLSTM + Attention model
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128):
        super(BiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        attn_weights = torch.softmax(self.attn(lstm_out).squeeze(-1), dim=1)
        context = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)
        out = self.fc(context)
        return self.sigmoid(out)

# Initialize and train model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMAttention(len(vocab), 100, 128).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

def evaluate_model(model, dev_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in dev_loader:
            texts = texts.to(device)
            outputs = model(texts).cpu().squeeze()
            preds = (outputs > 0.5).long()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    print(classification_report(all_labels, all_preds, target_names=["Not Hope", "Hope"]))

# Train and evaluate
train_model(model, train_loader, criterion, optimizer)
evaluate_model(model, dev_loader)

# Save model and vocab
torch.save(model.state_dict(), "bilstm_attention_model.pt")
with open("bilstm_vocab.json", "w") as f:
    json.dump(vocab, f)
print("✅ BiLSTM+Attention model and vocab saved.")



Epoch 1, Loss: 94.5772
Epoch 2, Loss: 66.6634
Epoch 3, Loss: 53.3557
Epoch 4, Loss: 43.0259
Epoch 5, Loss: 29.0137
Epoch 6, Loss: 15.5887
Epoch 7, Loss: 7.9172
Epoch 8, Loss: 5.2948
Epoch 9, Loss: 3.5993
Epoch 10, Loss: 2.0404
              precision    recall  f1-score   support

    Not Hope       0.79      0.83      0.81      1003
        Hope       0.80      0.75      0.78       899

    accuracy                           0.79      1902
   macro avg       0.79      0.79      0.79      1902
weighted avg       0.79      0.79      0.79      1902

✅ BiLSTM+Attention model and vocab saved.
